# 03 · Benchmark de modelos densos (calidad frente a coste CPU)

Lee el **smoke test** más reciente (`scripts/benchmark_dense_models.py --smoke-test`) y compara
los modelos por dimensión, throughput de codificación, latencia de query y RAM pico. No codifica
aquí: solo consume el reporte.

Genera el smoke test en el servidor:

```bash
uv run python scripts/benchmark_dense_models.py --smoke-test
```

In [ ]:
import json
from pathlib import Path

import pandas as pd

SMOKE_ROOT = Path("data/processed/reports/dense/smoke_tests")
runs = sorted(p for p in SMOKE_ROOT.glob("*") if (p / "report.json").is_file())
if not runs:
    raise FileNotFoundError(
        f"No hay smoke tests en {SMOKE_ROOT}. Ejecuta benchmark_dense_models.py --smoke-test."
    )
run = runs[-1]
report = json.loads((run / "report.json").read_text(encoding="utf-8"))
print(f"smoke_test_id: {report['smoke_test_id']} | threads={report.get('threads')}")

In [ ]:
df = pd.read_csv(run / "models.csv")
cols = [
    c
    for c in [
        "model_alias",
        "embedding_dimension",
        "doc_throughput_per_s",
        "query_latency_p50_s",
        "query_latency_p95_s",
        "peak_ram_mb",
        "warning",
    ]
    if c in df.columns
]
df[cols]

## Lectura

El smoke test no mide **calidad de retrieval** (eso es el benchmark formal, notebooks 04–05): aquí
se acota el **coste** por modelo en CPU para descartar candidatos inviables antes del benchmark.
Selecciona finalistas equilibrando dimensión/coste y guarda la figura final en `thesis/figures/`.